In [1]:
from backend.utils.gmsh_function        import *
from backend.utils.file_path            import *
from backend.utils.h_refinement_loop    import *
from backend.utils.ifa_meander_project_v2.algorithms import *

In [2]:
name = "ifa"
# Define the path to save the file
path = setup_save_file_paths(name)

In [3]:
a = 25 / 1000  # Width
b = 30 / 1000  # Height

terminal_a = 55 / 1000
terminal_b = b
x_t = np.array([-terminal_a + a, 0, 0, -terminal_a + a])
y_t = np.array([terminal_b, terminal_b, 0, 0])

largeur_piste    = a / 20          # Minimum value
distance_meandre = a / 20          # Value chosen based on the result from analyse_dist_meandre.ipynb

feed = 0.75 * b
feed_point = np.array([0, feed, 0])
mesh_size = 15 * largeur_piste

# Display values
print(f"Track width = {largeur_piste * 1000:.2f} mm")
print(f"Meander distance = {distance_meandre * 1000:.2f} mm")
print(f"Feed = {feed * 1000:.2f} mm")
print(f"Mesh size = {mesh_size * 1000:.2f} mm")

Track width = 1.25 mm
Meander distance = 1.25 mm
Feed = 22.50 mm
Mesh size = 18.75 mm


In [4]:
light_speed = 3e8
frequency = 1000e6   #868 MHZ
print(f"frequency = {frequency * 1e-06} MHz")
wavelength = light_speed / frequency
print(f"wavelength = {wavelength * 1000:.2f} mm")

mesh_size_by_wavelength = (wavelength) / 60
print(f"Mesh size by wavelength = {mesh_size_by_wavelength * 1000:.2f} mm")

frequency = 1000.0 MHz
wavelength = 300.00 mm
Mesh size by wavelength = 5.00 mm


In [5]:
initial_mesh_size = mesh_size

print(f"initial_mesh_size = {initial_mesh_size}")

initial_mesh_size = 0.01875


In [6]:
# Initialize Gmsh
gmsh.initialize()
gmsh.model.add("IFA_meander")
setup_performance_config()

# Create the IFA 
x, y, N, distance_meandre = ifa_creation(a, b, largeur_piste, distance_meandre)
x_m, y_m = trace_meander(x, y, largeur_piste)
feed_wid = largeur_piste                # The width of the track is the same everywhere.
feed_x = np.array([0, distance_meandre, distance_meandre, 0])
feed_y = np.array([feed + feed_wid/2, feed + feed_wid/2, feed -feed_wid/2, feed - feed_wid/2])

ifa_meander = rectangle_surface(x_m, y_m)
# print("tag of ifa_meander =", ifa_meander)

ifa_feed = rectangle_surface(feed_x, feed_y)
# print("tag of ifa_feed =", ifa_feed)

# Creation of the terminal
terminal = rectangle_surface(x_t, y_t)

# Fusion of the terminal and the meander
antenna_ifa_meander, _ = gmsh.model.occ.fuse([(2, ifa_meander)], [(2, terminal), (2, ifa_feed)])

# Synchronization and saving
gmsh.model.occ.synchronize()

generate_and_save_mesh(path, initial_mesh_size)

# gmsh.fltk.run()

# Close Gmsh
gmsh.finalize()

[PERFORMANCE] Gmsh configured to utilize 16 threads.
Geometry file saved in data/gmsh_files/ifa.brep successfully
--- Starting Mesh Optimization (Dim: 2) ---
--- Optimization Complete ---
Mesh file saved in data/gmsh_files/ifa.msh successfully


_ = radiation_algorithm(path, frequency, feed_point, excitation_unit_vector='x', gap_width=1.1*initial_mesh_size)

In [7]:
# Mesh refinement setup
grid_points, radius_vicinity = generate_embedded_grid(path.geo)
config = initialize_refinement_config(initial_mesh_size)
# view_grid_point(path.geo, grid_points)

run_radiation_refinement(config, grid_points, radius_vicinity, path, frequency, feed_point, excitation_unit_vector='x', gap_width=3*initial_mesh_size/2)

[PERFORMANCE] Gmsh configured to utilize 16 threads.
Object diagonal: 0.0627
Fixed spacing (10.0%): 0.0063
Generated 197 points.
Computed r_vicinity: 0.003519
Grid generation complete. Points shape: (197, 3)
Configuration initialized. Max iterations: 3
Mesh size array initialized with uniform size: 0.01875



>>> Starting Iteration 1/3
[PERFORMANCE] Gmsh configured to utilize 16 threads.
[PERFORMANCE] Gmsh configured to utilize 16 threads.
--- Starting Mesh Optimization (Dim: 2) ---
--- Optimization Complete ---
MATLAB file stored in data/antennas_mesh/ifa.mat successfully


Points refined in this step: 24
Total modified points in grid: 0
Size range: [0.004687, 0.018750]

>>> Starting Iteration 2/3
[PERFORMANCE] Gmsh configured to utilize 16 threads.
[PERFORMANCE] Gmsh configured to utilize 16 threads.
--- Starting Mesh Optimization (Dim: 2) ---
--- Optimization Complete ---
MATLAB file stored in data/antennas_mesh/ifa.mat successfully


Points refined in this step: 41
Total modified points in grid: 0
Size range: [0.001172, 0.018750]

>>> Starting Iteration 3/3
[PERFORMANCE] Gmsh configured to utilize 16 threads.
[PERFORMANCE] Gmsh configured to utilize 16 threads.
--- Starting Mesh Optimization (Dim: 2) ---
--- Optimization Complete ---
MATLAB file stored in data/antennas_mesh/ifa.mat successfully


Points refined in this step: 39
Total modified points in grid: 0
Size range: [0.000293, 0.018750]

>>> Refinement process completed.
